<a href="https://colab.research.google.com/github/Aaryan-Patel-27/BASIC_ML_MODELS/blob/main/ALL_SUPERVISED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import load_breast_cancer

# Load the breast_cancer dataset
X, y = load_breast_cancer(return_X_y=True)

print("Breast Cancer Dataset loaded successfully.")
print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")

Breast Cancer Dataset loaded successfully.
Shape of features (X): (569, 30)
Shape of target (y): (569,)


In [ ]:
import numpy as np
import pandas as pd

# Convert X to a Pandas DataFrame
X_df = pd.DataFrame(X)

# Introduce 5% missing values randomly in X_df
missing_percentage = 0.05
missing_count = int(X_df.size * missing_percentage)

# Get random indices to introduce missing values
np.random.seed(42) # for reproducibility
missing_row_indices = np.random.choice(X_df.shape[0], missing_count, replace=True)
missing_col_indices = np.random.choice(X_df.shape[1], missing_count, replace=True)

for i in range(missing_count):
    X_df.iloc[missing_row_indices[i], missing_col_indices[i]] = np.nan

print(f"Original X shape: {X.shape}")
print(f"X_df shape after converting and adding missing values: {X_df.shape}")
print(f"Number of missing values introduced: {X_df.isnull().sum().sum()}")
print("First 5 rows of X_df with potential missing values:")
print(X_df.head())

Original X shape: (569, 30)
X_df shape after converting and adding missing values: (569, 30)
Number of missing values introduced: 831
First 5 rows of X_df with potential missing values:
      0      1       2       3        4        5       6        7       8   \
0  17.99  10.38  122.80  1001.0  0.11840  0.27760  0.3001  0.14710  0.2419   
1    NaN  17.77  132.90  1326.0  0.08474  0.07864  0.0869  0.07017  0.1812   
2  19.69  21.25  130.00  1203.0  0.10960  0.15990  0.1974  0.12790  0.2069   
3  11.42  20.38   77.58   386.1  0.14250  0.28390  0.2414  0.10520  0.2597   
4  20.29  14.34  135.10  1297.0  0.10030  0.13280     NaN  0.10430  0.1809   

        9   ...     20     21      22      23      24      25      26      27  \
0  0.07871  ...  25.38  17.33  184.60  2019.0  0.1622  0.6656     NaN  0.2654   
1  0.05667  ...    NaN  23.41  158.80     NaN  0.1238  0.1866  0.2416  0.1860   
2  0.05999  ...  23.57  25.53  152.50  1709.0  0.1444  0.4245     NaN  0.2430   
3  0.09744  ...  14.9

In [ ]:
from collections import Counter

# Identify majority and minority classes in y
class_counts = Counter(y)
majority_class = max(class_counts, key=class_counts.get)
minority_class = min(class_counts, key=class_counts.get)

print(f"Original class distribution: {class_counts}")
print(f"Majority class: {majority_class}")
print(f"Minority class: {minority_class}")

# Create class imbalance: downsample the majority class
# Let's aim for a ratio where the majority class is 1.5 times the minority class (approx. 1:0.67 ratio)
# Current ratio is roughly 2:1 (357:212)

# Get indices for each class
majority_indices = np.where(y == majority_class)[0]
minority_indices = np.where(y == minority_class)[0]

# Determine the target size for the majority class to achieve imbalance
# Let's reduce the majority class to be 1.5 times the minority class size
target_majority_size = int(len(minority_indices) * 1.5)

# Randomly select a subset of majority class indices to keep
np.random.seed(42) # for reproducibility
kept_majority_indices = np.random.choice(majority_indices, target_majority_size, replace=False)

# Combine the kept majority indices and all minority indices
selected_indices = np.concatenate([kept_majority_indices, minority_indices])

# Shuffle the indices to mix the classes
np.random.shuffle(selected_indices)

# Apply the selected indices to X_df and y
X_df_imbalanced = X_df.iloc[selected_indices].reset_index(drop=True)
y_imbalanced = y[selected_indices]

print(f"\nShape of X_df after imbalance: {X_df_imbalanced.shape}")
print(f"Shape of y after imbalance: {y_imbalanced.shape}")
print(f"New class distribution in y: {Counter(y_imbalanced)}")

Original class distribution: Counter({np.int64(1): 357, np.int64(0): 212})
Majority class: 1
Minority class: 0

Shape of X_df after imbalance: (530, 30)
Shape of y after imbalance: (530,)
New class distribution in y: Counter({np.int64(1): 318, np.int64(0): 212})


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_df_imbalanced,
                                                    y_imbalanced,
                                                    test_size=0.3,
                                                    random_state=42,
                                                    stratify=y_imbalanced) # Stratify to maintain class distribution

print("Data split into training and testing sets successfully.")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# 2. Handle missing values using median imputation
imputer = SimpleImputer(strategy='median')

# Fit the imputer on X_train and transform both X_train and X_test
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Convert back to DataFrame to maintain column names for easier debugging/inspection if needed
X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns)

print("\nMissing values imputed successfully using median strategy.")
print(f"Number of missing values in X_train_imputed: {X_train_imputed.isnull().sum().sum()}")
print(f"Number of missing values in X_test_imputed: {X_test_imputed.isnull().sum().sum()}")

# 3. Scale numerical features using StandardScaler
scaler = StandardScaler()

# Fit the scaler on imputed X_train and transform both imputed X_train and X_test
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Convert back to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("\nFeatures scaled successfully using StandardScaler.")
print("First 5 rows of X_train_scaled after preprocessing:")
print(X_train_scaled.head())

Data split into training and testing sets successfully.
X_train shape: (371, 30), y_train shape: (371,)
X_test shape: (159, 30), y_test shape: (159,)

Missing values imputed successfully using median strategy.
Number of missing values in X_train_imputed: 0
Number of missing values in X_test_imputed: 0

Features scaled successfully using StandardScaler.
First 5 rows of X_train_scaled after preprocessing:
         0         1         2         3         4         5         6   \
0  0.552305 -0.904389  0.559157  0.398765 -0.138920  0.653854  0.375520   
1  1.135052  0.142821  0.923274  0.970233 -0.930997 -0.745115 -0.224701   
2 -0.177646 -1.041690 -0.839661 -0.785336 -0.453132 -0.458235 -0.365305   
3 -1.077564 -0.988166 -1.101478 -0.943942 -0.647333 -1.229590 -1.146845   
4 -0.352166 -1.293020 -0.441732 -0.396084 -0.967364 -1.343101 -1.089200   

         7         8         9   ...        20        21        22        23  \
0  0.545437 -0.079685 -0.447305  ...  0.509021 -0.624198  0.48

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Instantiate the models
logistic_regression_model = LogisticRegression(random_state=42)
svc_model = SVC(random_state=42)
decision_tree_model = DecisionTreeClassifier(random_state=42)

print("Models instantiated successfully.")

# Train each model
logistic_regression_model.fit(X_train_scaled, y_train)
svc_model.fit(X_train_scaled, y_train)
decision_tree_model.fit(X_train_scaled, y_train)

print("Models trained successfully.")

# Make predictions on the test data
y_pred_lr = logistic_regression_model.predict(X_test_scaled)
y_pred_svc = svc_model.predict(X_test_scaled)
y_pred_dt = decision_tree_model.predict(X_test_scaled)

print("Predictions made successfully for all models.")

Models instantiated successfully.
Models trained successfully.
Predictions made successfully for all models.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# --- Evaluate Logistic Regression Model ---
print("\n--- Logistic Regression Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

# --- Evaluate Support Vector Machine (SVC) Model ---
print("\n--- SVC Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svc):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svc):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_svc):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_svc):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svc))

# --- Evaluate Decision Tree Classifier Model ---
print("\n--- Decision Tree Classifier Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_dt):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))


--- Logistic Regression Model Evaluation ---
Accuracy: 0.9874
Precision: 0.9895
Recall: 0.9895
F1-Score: 0.9895
Confusion Matrix:
 [[63  1]
 [ 1 94]]

--- SVC Model Evaluation ---
Accuracy: 0.9811
Precision: 0.9894
Recall: 0.9789
F1-Score: 0.9841
Confusion Matrix:
 [[63  1]
 [ 2 93]]

--- Decision Tree Classifier Model Evaluation ---
Accuracy: 0.9560
Precision: 0.9400
Recall: 0.9895
F1-Score: 0.9641
Confusion Matrix:
 [[58  6]
 [ 1 94]]


## Summary:

### Data Analysis Key Findings

*   The `breast_cancer` dataset, initially comprising 569 samples with 30 features, was loaded successfully.
*   **Data Manipulation**:
    *   Missing values were artificially introduced into 5% of the dataset, resulting in 831 `NaN` values.
    *   Class imbalance was deliberately created by downsampling the majority class (class 1) from 357 to 318 samples, while keeping the minority class (class 0) at 212 samples. The final imbalanced dataset had 530 samples with a class distribution of 318 for class 1 and 212 for class 0.
*   **Preprocessing**:
    *   The imbalanced dataset was split into training (371 samples) and testing (159 samples) sets using `stratify` to preserve the class distribution (approximately 64 samples of class 0 and 95 samples of class 1 in the test set).
    *   Missing values were successfully handled using median imputation.
    *   Numerical features were scaled using `StandardScaler`.
*   **Model Performance Evaluation on Imbalanced Data**:
    *   **Logistic Regression**: Achieved high overall performance with an F1-score of 0.9895 (for class 1) and an F1-score of 0.9844 (for class 0), indicating strong performance on both classes. It correctly classified 63 out of 64 minority class samples (Recall=0.9844) and had only 1 false positive and 1 false negative.
        *   Accuracy: 0.9874
        *   Precision (Class 1): 0.9895
        *   Recall (Class 1): 0.9895
        *   F1-Score (Class 1): 0.9895
        *   Confusion Matrix: `[[63, 1], [1, 94]]`
    *   **Support Vector Machine (SVC)**: Performed very similarly to Logistic Regression, with a slightly lower overall F1-score for class 1 (0.9841) and for class 0 (0.9767) compared to Logistic Regression. It had 2 false negatives for class 1. It correctly classified 63 out of 64 minority class samples (Recall=0.9844).
        *   Accuracy: 0.9811
        *   Precision (Class 1): 0.9894
        *   Recall (Class 1): 0.9789
        *   F1-Score (Class 1): 0.9841
        *   Confusion Matrix: `[[63, 1], [2, 93]]`
    *   **Decision Tree Classifier**: Showed the lowest overall performance among the three models, particularly in terms of F1-score for class 0 (0.9431) and accuracy (0.9560). While its recall for class 1 was high (0.9895), its recall for class 0 was notably lower (0.9063), incorrectly classifying 6 minority class samples as majority. It also had 6 false positives.
        *   Accuracy: 0.9560
        *   Precision (Class 1): 0.9400
        *   Recall (Class 1): 0.9895
        *   F1-Score (Class 1): 0.9641
        *   Confusion Matrix: `[[58, 6], [1, 94]]`

### Insights or Next Steps

*   **Model Suitability**: Both Logistic Regression and SVC models demonstrated superior performance on this imbalanced breast cancer dataset, achieving very high F1-scores and recall for both majority and minority classes. Logistic Regression slightly edged out SVC in terms of overall balance, especially concerning the minority class. Decision Tree, without specific tuning, struggled more with the minority class recall compared to the other two.
*   **Further Optimization**: For the Decision Tree, hyperparameter tuning (e.g., `max_depth`, `min_samples_leaf`) and specific techniques for imbalanced data (e.g., `class_weight='balanced'`) could significantly improve its performance, especially in reducing false positives and improving minority class recall. For all models, exploring more advanced resampling techniques (e.g., SMOTE, ADASYN) or ensemble methods (e.g., Random Forest, Gradient Boosting) could potentially lead to even more robust results on imbalanced datasets.
